# CyberGYM SFT — Kaggle T4
Cell 1: clone + download checkpoint + prepare SFT data (~15 min)
Cell 2: SFT training (~20-40 min)
Cell 3: test chat generation


In [ ]:
import os, subprocess, sys

# --- clone ---
if not os.path.isdir("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "https://github.com/ziyy1337/train-llm-from-scratch", "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
print("CLONE OK")

# --- install deps ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jaxtyping", "beartype", "huggingface_hub"], check=True)
print("DEPS OK")

# --- download checkpoint from HuggingFace ---
ckpt = "/kaggle/working/500m_cybergym.pt"
if not os.path.exists(ckpt):
    subprocess.run(["huggingface-cli", "download", "ziyy1337/monocyber-checkpoint", "500m_cybergym.pt", "--local-dir", "/kaggle/working/"], check=True)
print("CHECKPOINT OK:", os.path.getsize(ckpt))

# --- prepare SFT data (downloads Alpaca + Dolly + GSM8K) ---
subprocess.run(["python", "scripts/prepare_sft_data.py", "--context_length", "512", "--out_dir", "data"], check=True)
print("SFT DATA OK")



In [ ]:
import os
os.chdir("/kaggle/working/repo")
!python scripts/train_sft.py --arch modern --vocab_size 154880 --context_length 512 --n_embed 1024 --n_head 8 --n_blocks 28 --n_kv_head 2 --tie_embeddings True --pretrained_ckpt /kaggle/working/500m_cybergym.pt --data_path data/sft_packed.h5 --dev_path data/sft_dev_packed.h5 --out_ckpt /kaggle/working/models/sft.pt --device cuda --amp_dtype fp16 --batch_size 8 --grad_accum 2 --epochs 1 --max_steps 500 --lr 1e-5 --warmup_steps 50 --eval_steps 100 --save_every 250 --lora_rank 16 --lora_alpha 32



In [ ]:
import os
os.chdir("/kaggle/working/repo")
!python scripts/generate_text.py --model_path /kaggle/working/models/sft.pt --input_text "What is 2+2?" --max_new_tokens 100

